In [ ]:
# ---- x7: can Saner Turhaner's scorer (kit sanpier/casmi-spectrum-model-v1) train on Kaggle's TPU v3-8 with PyTorch/XLA? ----
# The notebook process never initialises the TPU runtime (one process holds the device); every test runs tpu_train.py
# (kaggle/forks/tpu_train.py, embedded below) as a subprocess: 1) one core, 2 short epochs; 2) xmp.spawn, one seed per core.
import json, os, subprocess, sys, time
from pathlib import Path
W = Path('/kaggle/working'); T0 = time.time()
def log(*a): print(f'[{time.time() - T0:7.1f}s]', *a, flush=True)
os.environ.setdefault('PJRT_DEVICE', 'TPU')
rep = {'cpu_count': os.cpu_count(), 'env': {k: v for k, v in os.environ.items() if any(s in k for s in ('TPU', 'XLA', 'PJRT'))}}
try: rep['mem_gb'] = round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1)
except Exception as e: rep['mem_error'] = repr(e)
pk = subprocess.run([sys.executable, '-m', 'pip', 'list'], capture_output=True, text=True).stdout
rep['packages'] = [l for l in pk.splitlines() if any(s in l.lower() for s in ('torch', 'jax', 'tensorflow', 'libtpu', 'numpy', 'pandas', 'pyarrow'))]
q = subprocess.run([sys.executable, '-c', "import torch, torch_xla, torch_xla.runtime as xr; print(torch.__version__, torch_xla.__version__, xr.device_type(), xr.global_runtime_device_count())"],
                   capture_output=True, text=True, timeout=600)
rep['xla_query'] = dict(rc=q.returncode, out=q.stdout[-2000:], err=q.stderr[-3000:])
log(json.dumps(rep, indent=1))
SRC = "\"\"\"Full-data refits of Saner Turhaner's scorer (kit sanpier/casmi-spectrum-model-v1, unchanged train.py) on Kaggle\nTPU v3-8 with PyTorch/XLA. Written to /kaggle/working by the n4/x7 notebooks and run as a subprocess (xmp.spawn needs\nan importable worker, which a notebook's __main__ is not).\n\n    python tpu_train.py --mode single --k0 301 --epochs 2 --steps 100          # one TPU core, timing probe\n    python tpu_train.py --mode spawn  --k0 301 --epochs 64 --save             # one seed per TPU core, k0 + ordinal\n\nSeed k uses config seed 20261023 + 1000*k, like scripts/local_seeds.py. With --save the refit goes to\n/kaggle/working/seeds/casmi26_seed_<k>_e<epochs>.pt. Each replica writes reports/replica_<k>.json.\n\"\"\"\nimport argparse\nimport json\nimport os\nimport shutil\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nBASE_SEED = 20261023\nW = Path(\"/kaggle/working\")\nINP = Path(\"/kaggle/input\")\n\n\ndef args():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--mode\", choices=[\"single\", \"spawn\"], default=\"single\")\n    ap.add_argument(\"--k0\", type=int, default=301)\n    ap.add_argument(\"--epochs\", type=int, default=2)\n    ap.add_argument(\"--steps\", type=int, default=500)\n    ap.add_argument(\"--save\", action=\"store_true\")\n    ap.add_argument(\"--hours\", type=float, default=8.0, help=\"per-replica training deadline\")\n    return ap.parse_args()\n\n\nA = args()\n\n\ndef worker(index):\n    t0 = time.time()\n    k = A.k0 + index\n    rep = dict(k=k, index=index, pid=os.getpid())\n    (W / \"reports\").mkdir(exist_ok=True)\n    try:\n        import torch_xla.core.xla_model as xm\n        dev = xm.xla_device()\n        rep[\"device\"] = str(dev)\n        kit = next(p.parent for p in INP.rglob(\"train.py\") if (p.parent / \"model.py\").is_file())\n        cache = next(p.parent for p in INP.rglob(\"peaks.bin\") if (p.parent / \"fingerprints.npy\").is_file())\n        sys.path.insert(0, str(kit))\n        import train as T\n\n        class AdamW(torch.optim.AdamW):\n            # Cut the lazy XLA graph once per optimizer step.\n            def step(self, *a, **kw):\n                out = super().step(*a, **kw)\n                xm.mark_step()\n                return out\n\n        torch.optim.AdamW = AdamW\n        cfg = json.loads((kit / \"config.json\").read_text())\n        cfg.update(work=str(cache), steps_per_epoch=A.steps)\n        data = T.Data(cfg)\n        data.x = np.array(data.x)\n        data.meta = np.array(data.meta)\n        data.fp = np.array(data.fp)\n        rep[\"load_seconds\"] = round(time.time() - t0, 1)\n        out = W / \"runs\" / f\"refit_{k}\"\n        out.mkdir(parents=True, exist_ok=True)\n        c = dict(cfg, seed=BASE_SEED + 1000 * k)\n        t = time.time()\n        res = T.train_stage(data, c, out, dev, refit=True, epochs=A.epochs,\n                            deadline=time.monotonic() + A.hours * 3600)\n        rep.update(res, train_seconds=round(time.time() - t, 1),\n                   history=json.loads((out / \"refit_history.json\").read_text()))\n        if A.save:\n            (W / \"seeds\").mkdir(exist_ok=True)\n            shutil.move(str(out / \"final_model.pt\"), str(W / \"seeds\" / f\"casmi26_seed_{k}_e{A.epochs}.pt\"))\n        shutil.rmtree(out, ignore_errors=True)\n    except Exception:\n        rep[\"error\"] = traceback.format_exc()[-4000:]\n    rep[\"seconds\"] = round(time.time() - t0, 1)\n    (W / \"reports\" / f\"replica_{k}.json\").write_text(json.dumps(rep, indent=2, default=str))\n    print(\"REPLICA\", json.dumps({k: v for k, v in rep.items() if k != \"history\"}, default=str), flush=True)\n\n\ndef _mp_fn(index):\n    # Module level so spawned children can import it.\n    import torch_xla.runtime as xr\n    worker(xr.global_ordinal())\n\n\nif __name__ == \"__main__\":\n    if A.mode == \"single\":\n        worker(0)\n    else:\n        import torch_xla.distributed.xla_multiprocessing as xmp\n        xmp.spawn(_mp_fn, args=())\n"
(W / 'tpu_train.py').write_text(SRC)
tests = [('single', ['--mode', 'single', '--epochs', '2', '--steps', '100', '--k0', '301'], 1800),
         ('spawn', ['--mode', 'spawn', '--epochs', '2', '--steps', '100', '--k0', '311'], 2700)]
for name, cmd, timeout in tests:
    t = time.time()
    try:
        p = subprocess.run([sys.executable, str(W / 'tpu_train.py'), *cmd], capture_output=True, text=True, timeout=timeout)
        rep[name] = dict(rc=p.returncode, seconds=round(time.time() - t), stdout=p.stdout[-8000:], stderr=p.stderr[-8000:])
    except subprocess.TimeoutExpired as e:
        rep[name] = dict(timeout=timeout, stdout=str(e.stdout)[-8000:], stderr=str(e.stderr)[-8000:])
    log(name, 'rc', rep[name].get('rc'), 'seconds', rep[name].get('seconds'))
    print(rep[name]['stdout'][-4000:]); print(rep[name]['stderr'][-4000:])
    (W / 'x7_report.json').write_text(json.dumps(rep, indent=1, default=str))
reports = {p.name: json.loads(p.read_text()) for p in sorted((W / 'reports').glob('*.json'))} if (W / 'reports').is_dir() else {}
for n, r in reports.items():
    print('REPORT', n, r.get('device'), r.get('load_seconds'), r.get('train_seconds'), [h.get('seconds') for h in r.get('history', [])], (r.get('error') or '')[-600:])
log('X7_DONE')
